# Vortex-50M-16k — Tokenizer → Train → Push

A **49,844,992 parameter** decoder-only Transformer, from scratch, on an English-only **16,384-token** vocabulary.

```
49,844,992 params  |  512d x 18L  |  8Q/2KV GQA  |  vocab 16,384 (tied)
17% embedding  /  83% transformer layers
```

**Data**: `HuggingFaceTB/smollm-corpus`, streamed (never fully downloaded) · **2B tokens**
**Hardware**: RTX 4090, 24 GB — batch size auto-detected at runtime

---

## The three phases

| Phase | What happens | Runtime (approx) |
|---|---|---|
| **1. Tokenizer** | Train a 16,384-token BPE on the corpus, measure compression, stream+tokenize 2B tokens into shards | 30 min – 3 h |
| **2. Model** | Verify the architecture, then pretrain on the shards | days |
| **3. Push** | Push the final weights + tokenizer to the Hub | minutes |

Phase 1 and 2 are independent of each other, but **2 cannot start until 1 finishes** — the shards must exist, and they must have been built with the *same* tokenizer the model is configured for.

**Set `RUN_ALL = True` in cell 11 to execute all three phases in one go**, or run them one at a time by executing the cells you want.

---
## 0. Setup

In [ ]:
import os, sys, json, subprocess, time, math, shutil
from pathlib import Path

def run(cmd, check=True, cwd=None, env=None):
    print(f"\n{'-'*70}\n$ {cmd}\n{'-'*70}", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, check=check, cwd=cwd, env=env)
    if check and r.returncode != 0:
        raise SystemExit(f"FAILED ({r.returncode}): {cmd}")
    print(f"[{time.time()-t0:.0f}s]", flush=True)
    return r

print("Python:", sys.version.split()[0])
run("nvidia-smi", check=False)

In [ ]:
CLONE_REPO = True
GIT_REPO_URL = "https://github.com/OEvortex/VTX-300M.git"
REPO_DIR = Path("/content/VTX-300M")

if CLONE_REPO and not REPO_DIR.exists():
    run(f"git clone {GIT_REPO_URL} {REPO_DIR}")
SRC_DIR = (REPO_DIR if CLONE_REPO else Path.cwd()) / "src"
assert (SRC_DIR / "pretrain.py").exists(), f"src/pretrain.py not found under {SRC_DIR}"
print("Source:", SRC_DIR)

In [ ]:
INSTALL_TORCH = True
INSTALL_EVAL = True

run(f"{sys.executable} -m pip install -U pip setuptools wheel")
if INSTALL_TORCH:
    run(f"{sys.executable} -m pip install -U torch --index-url https://download.pytorch.org/whl/cu124")

deps = ["transformers>=4.45.0", "huggingface_hub>=0.24.0", "hf_transfer",
        "safetensors", "numpy", "scipy", "trackio", "tokenizers", "datasets"]
if INSTALL_EVAL:
    deps.append("accelerate")
run(f"{sys.executable} -m pip install -U " + " ".join(deps))

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
from getpass import getpass
from huggingface_hub import login, whoami, create_repo

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HF_TOKEN: ")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
login(token=HF_TOKEN, add_to_git_credential=False)
print(whoami(token=HF_TOKEN))

---
## Configuration

`RUN_ALL = True` in the next cell executes **all three phases**. Set it to `False`
to run them individually.

`vortex-50m-16k` is the 512d × **18-layer** shape. The 16K vocab frees 8.4M
parameters versus a 32K vocab, and the budget it frees goes into *depth* rather
than a wider lookup table.

In [ ]:
# ======================================================================
#  RUN EVERYTHING
# ======================================================================
RUN_ALL = True
# If the run is interrupted, re-run the notebook: every phase is idempotent
# and will skip work that is already done.

# -- Model ----------------------------------------------------------------
ARCH            = "vortex-50m-16k"    # 49,844,992 params | 512d x 18L | 8Q/2KV
TOKENIZER_VOCAB = 16_384
CONTEXT         = 2048

# -- Repos ----------------------------------------------------------------
MODEL_REPO_ID    = "VTXAI/vortex-50m-16k"
TOKENIZER_REPO   = "VTXAI/vortex-tok-16k"
TRACKIO_SPACE_ID = "VTXAI/vortex-50m-trackio"   # None disables Trackio
TRACKIO_PROJECT  = "vortex-50m-16k"

# -- Data: streamed from the Hub ------------------------------------------
HF_REPO    = "HuggingFaceTB/smollm-corpus"
HF_CONFIGS = ["cosmopedia-v2", "fineweb-edu-dedup"]  # python-edu has NO text column
TOKENIZER_DOCS = 120_000       # documents used to learn the 16K vocabulary
TARGET_TOKENS  = 2_000_000_000 # 2B-token training budget

# -- Paths ----------------------------------------------------------------
TOKENIZER_DIR = "/content/vortex-tok-16k"
DATA_DIR      = "/content/data16k"
SAVE_DIR      = "/content/vortex_50m_ckpt"

# -- Training -------------------------------------------------------------
TARGET_BATCH = 32              # effective batch in sequences
AUTO_BATCH   = True            # probe the 4090 for the largest batch that fits
LR, MIN_LR   = 6e-4, 6e-5
PUSH_EVERY, LOG_EVERY = 3000, 25
COMPILE = True
VAL_EVERY, VAL_TOKENS = 500, 2_000_000   # held-out overfit guard

TOK_PER_STEP = TARGET_BATCH * CONTEXT
STEPS   = TARGET_TOKENS // TOK_PER_STEP
WARMUP  = max(200, STEPS // 100)
RUN_TOKENS = f"{TARGET_TOKENS/1e9:.0f}B"

print(f"ARCH        {ARCH}  ({TARGET_TOKENS/1e9:.0f}B token budget)")
print(f"tokens/step {TOK_PER_STEP:,}   steps {STEPS:,}   warmup {WARMUP:,}")
print(f"phases      {'ALL (1,2,3)' if RUN_ALL else 'individual — run the cells you want'}")

---
# PHASE 1 — Tokenizer

The Qwen3 151,670-token vocab costs **77.7M** parameters as an embedding table at
512d — **156% of the entire 50M budget** before a single layer exists. A 16K
English BPE costs **8.4M**, and the leftover buys a deeper 18-layer stack.

`cosmopedia-v2` (synthetic textbooks) and `fineweb-edu-dedup` (filtered web) are
both English educational text. `python-edu` is **excluded** — its schema has no
`text` column, only blob ids, so it cannot be streamed as raw text.

In [ ]:
def phase1_tokenizer():
    """Train the 16K vocabulary, measure it, then stream+tokenize 2B tokens."""
    from tokenizers import Tokenizer

    # -- 1a. train the vocabulary -----------------------------------------
    tok_path = Path(TOKENIZER_DIR) / "tokenizer.json"
    if tok_path.exists() and not FORCE_RETRAIN_TOKENIZER:
        print(f"[phase1] tokenizer exists at {TOKENIZER_DIR} — skipping training")
    else:
        run(
            f"{sys.executable} train_tokenizer.py --hf-repo {HF_REPO} "
            f"--hf-configs {' '.join(HF_CONFIGS)} --max-docs {TOKENIZER_DOCS} "
            f"--out {TOKENIZER_DIR} --vocab-size {TOKENIZER_VOCAB}",
            cwd=SRC_DIR,
        )

    tok = Tokenizer.from_file(str(tok_path))
    actual = tok.get_vocab_size()
    print(f"\n[phase1] tokenizer vocab = {actual:,} (requested {TOKENIZER_VOCAB:,})")
    if actual != TOKENIZER_VOCAB:
        print("[phase1] NOTE: pretrain.py adopts the tokenizer's real vocab — that is")
        print("         correct, but the parameter count shifts slightly.")

    # -- 1b. measure compression on THIS corpus ---------------------------
    run(
        f"{sys.executable} train_tokenizer.py --out {TOKENIZER_DIR} --stats "
        f"--hf-repo {HF_REPO} --hf-configs {' '.join(HF_CONFIGS)} --stats-docs 2000",
        cwd=SRC_DIR, check=False,
    )

    # -- 1c. stream + tokenize up to the 2B budget ------------------------
    existing = sorted(Path(DATA_DIR).glob("*.bin")) if Path(DATA_DIR).exists() else []
    if existing and not FORCE_RETOKENIZE:
        print(f"[phase1] {len(existing)} shard(s) already in {DATA_DIR} — skipping retokenize")
    else:
        run(
            f"{sys.executable} retokenize.py --hf-repo {HF_REPO} "
            f"--hf-configs {' '.join(HF_CONFIGS)} --tokenizer {TOKENIZER_DIR} "
            f"--out {DATA_DIR} --max-tokens {TARGET_TOKENS} "
            f"--tokens-per-shard 100000000",
            cwd=SRC_DIR,
        )

    shards = sorted(str(p) for p in Path(DATA_DIR).glob("*.bin"))
    if not shards:
        raise SystemExit("[phase1] no .bin shards produced — check the stream output above")
    on_disk = sum(Path(s).stat().st_size for s in shards) // 4
    print(f"\n[phase1] {len(shards)} shard(s), {on_disk/1e9:.2f}B tokens on disk")
    if on_disk < TARGET_TOKENS * 0.95:
        print(f"[phase1] WARN: {on_disk/1e9:.2f}B is below the {RUN_TOKENS} target")
    return shards

FORCE_RETRAIN_TOKENIZER = False
FORCE_RETOKENIZE         = False

if RUN_ALL:
    shards = phase1_tokenizer()
else:
    print("RUN_ALL = False — run the Phase 1 cell below to build the shards.")
    shards = sorted(str(p) for p in Path(DATA_DIR).glob("*.bin")) if Path(DATA_DIR).exists() else []

---
# PHASE 2 — Model

**Optimizer**: fused **AdamW** — decoupled weight decay, none on 1-D norm gains,
fused CUDA kernel. Lion's `sign(momentum)` discards gradient magnitude, which is a
bad trade here: optimizer state is only ~0.4 GB of your 24 GB.

**Batch size**: `AUTO_BATCH=True` probes the 4090 for the largest batch that fits
(90% of the ceiling), then sets `grad_accum` to reach an effective batch of 32.

**Overfit guard**: every `VAL_EVERY` steps it scores a held-out tail of the corpus
and prints the train/val gap. Expect step 0 near `ln(16384) = 9.7` falling toward ~3.

In [ ]:
def phase2_train(shards):
    """Verify the architecture, then pretrain on the shards."""
    if not shards:
        raise SystemExit("[phase2] no shards — run Phase 1 first.")

    # -- 2a. architecture verification (cheap; gates a days-long run) -------
    r = run(f"{sys.executable} verify_arch.py", cwd=SRC_DIR, check=False)
    if r.returncode != 0:
        raise SystemExit("[phase2] architecture verification FAILED — do not train.")
    print("[phase2] architecture verified")

    # -- 2b. train ----------------------------------------------------------
    env = os.environ.copy()
    if TRACKIO_SPACE_ID:
        env["TRACKIO_SPACE_ID"] = TRACKIO_SPACE_ID
        env["TRACKIO_PROJECT"] = TRACKIO_PROJECT

    cmd = (
        f"{sys.executable} pretrain.py --arch {ARCH} --tokenizer {TOKENIZER_DIR} "
        f"--shards {' '.join(shards)} --block {CONTEXT} "
        f"--steps {STEPS} --warmup {WARMUP} --lr {LR} --min-lr {MIN_LR} "
        f"--auto-batch --target-batch {TARGET_BATCH} "
        f"--val-every {VAL_EVERY} --val-tokens {VAL_TOKENS} "
        f"--save-dir {SAVE_DIR} --hub-repo {MODEL_REPO_ID} "
        f"--push-every {PUSH_EVERY} --log-every {LOG_EVERY}"
    )
    if not COMPILE:
        cmd += " --no-compile"
    print("\n[phase2] training command:\n ", cmd, "\n")

    t0 = time.time()
    run(cmd, cwd=SRC_DIR, env=env)
    print(f"\n[phase2] training finished in {(time.time()-t0)/3600:.2f} h")


if RUN_ALL:
    phase2_train(shards)
else:
    print("RUN_ALL = False — run the Phase 2 cell below to train.")

---
# PHASE 3 — Push

Pushes the final weights **and** the tokenizer to the Hub, so the model is
loadable with a single `from_pretrained` call. The tokenizer must travel with the
model — the ids in the weights are meaningless without it.

In [ ]:
def latest_checkpoint(save_dir):
    root = Path(save_dir)
    steps = []
    for p in root.glob("step_*"):
        if p.is_dir():
            try: steps.append((int(p.name.split("_", 1)[1]), p))
            except Exception: pass
    return sorted(steps)[-1][1] if steps else None

LATEST = latest_checkpoint(SAVE_DIR)
print("Latest checkpoint:", LATEST)

if INSTALL_EVAL and LATEST is not None:
    run(f"{sys.executable} eval_benchmarks.py --ckpt {LATEST} "
        f"--tokenizer {TOKENIZER_DIR} --limit 500", cwd=SRC_DIR, check=False)

In [ ]:
def phase3_push():
    """Push the trained model and its tokenizer to the Hub."""
    from huggingface_hub import HfApi

    latest = latest_checkpoint(SAVE_DIR)
    if latest is None:
        raise SystemExit("[phase3] no checkpoint found — nothing to push")

    create_repo(MODEL_REPO_ID, token=HF_TOKEN, private=False, exist_ok=True)
    api = HfApi(token=HF_TOKEN)

    print(f"[phase3] pushing {latest.name} -> {MODEL_REPO_ID}")
    api.upload_folder(folder_path=str(latest), repo_id=MODEL_REPO_ID,
                      commit_message="final weights", token=HF_TOKEN)

    # The tokenizer is not optional — without it the token ids in the
    # checkpoint are unreadable.
    print(f"[phase3] pushing tokenizer -> {MODEL_REPO_ID}")
    api.upload_folder(folder_path=TOKENIZER_DIR, repo_id=MODEL_REPO_ID,
                      commit_message="tokenizer", token=HF_TOKEN)

    # Also publish it standalone so other projects can reuse the vocabulary.
    create_repo(TOKENIZER_REPO, token=HF_TOKEN, exist_ok=True)
    api.upload_folder(folder_path=TOKENIZER_DIR, repo_id=TOKENIZER_REPO,
                      commit_message="english 16k tokenizer", token=HF_TOKEN)

    print(f"\n[phase3] done -> {MODEL_REPO_ID}\n         and -> {TOKENIZER_REPO}")


if RUN_ALL:
    phase3_push()
else:
    print("RUN_ALL = False — run the Phase 3 cell above to push.")